# Lesson 7: Adding an MCP Server (Context7)

The Model Context Protocol (MCP) lets you plug external tool servers into your agent without writing any tool code.

We'll connect **Context7** — an MCP server that gives your agent access to up-to-date documentation for any programming library.

In [ ]:
%pip install openai-agents mermaid-py --upgrade --quiet

In [ ]:
import os
from getpass import getpass

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

MODEL = "gpt-5.4-mini"

<div style='margin-top: 20px; margin-bottom: 20px; text-align: center; display: flex; flex-direction: column; align-items: center;'>
<img src='./images/lesson_3_file_explorer.png' width='600' /> 
</div>

**Key idea**: Hand-written tools give you full control. MCP servers let you plug in capabilities from the ecosystem without writing tool code.

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## Connecting to Context7 via MCP

We use `MCPServerStdio` to launch the Context7 MCP server as a subprocess.
The agent automatically discovers the tools the server provides (`resolve-library-id`, `query-docs`).

In [4]:
async def main():
    async with MCPServerStdio(
        name="Context7",
        params={
            "command": "npx",
            "args": ["-y", "@upstash/context7-mcp"],
        },
    ) as context7_server:
        agent = Agent(
            name="Docs Assistant",
            instructions="You are a coding assistant with access to live documentation via Context7. When the user asks about a library, look up current docs and code examples.",
            model=MODEL,
            mcp_servers=[context7_server],
        )

        result = await Runner.run(
            agent,
            "How do I create a basic FastAPI app with a GET endpoint? Use context7 to look up the latest docs.",
        )
        print(result.final_output)


await main()

Here’s the simplest FastAPI app with a GET endpoint, based on the latest FastAPI docs:

```python
from fastapi import FastAPI

app = FastAPI()


@app.get("/")
async def root():
    return {"message": "Hello World"}
```

### What it does
- Creates a FastAPI app with `app = FastAPI()`
- Defines a GET route at `/`
- Returns JSON: `{"message": "Hello World"}`

### Run it
Save it as `main.py`, then start the dev server:

```bash
pip install --upgrade fastapi
fastapi dev main.py
```

### Test it
Open:
- `http://127.0.0.1:8000/`

You should see:

```json
{"message":"Hello World"}
```

If you want, I can also show:
- a version with `uvicorn`
- an endpoint with path params like `/items/{item_id}`
- how to add automatic docs at `/docs`


## How It Works

1. `MCPServerStdio` launches `npx @upstash/context7-mcp` as a child process
2. The SDK discovers the server's tools via the MCP protocol
3. The agent sees these tools alongside any hand-written tools
4. When the agent calls an MCP tool, the SDK routes the call to the server
5. The server returns results, which flow back into the agent loop

You can combine MCP servers with hand-written tools:

In [5]:
import subprocess
from agents import function_tool


@function_tool
def run_command(command: str) -> str:
    """Run a shell command and return its output."""
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=30)
        output = result.stdout
        if result.stderr:
            output += f"\nSTDERR:\n{result.stderr}"
        return output if output.strip() else "(no output)"
    except Exception as e:
        return f"Error: {e}"


async def main_with_tools():
    async with MCPServerStdio(
        name="Context7",
        params={
            "command": "npx",
            "args": ["-y", "@upstash/context7-mcp"],
        },
    ) as context7_server:
        agent = Agent(
            name="Full Coding Assistant",
            instructions="You are a coding assistant with hand-written tools and Context7 documentation lookup.",
            model=MODEL,
            tools=[run_command],
            mcp_servers=[context7_server],
        )

        result = await Runner.run(
            agent,
            "Look up how to use Python's pathlib to list all .txt files in a directory. Show me the code.",
        )
        print(result.final_output)


await main_with_tools()

Use `pathlib.Path.glob()` for files in one directory, or `Path.rglob()` to search recursively.

```python
from pathlib import Path

# Non-recursive: all .txt files in the given directory
directory = Path("path/to/your/directory")
txt_files = list(directory.glob("*.txt"))

for file in txt_files:
    print(file)
```

Recursive version:

```python
from pathlib import Path

directory = Path("path/to/your/directory")
txt_files = list(directory.rglob("*.txt"))

for file in txt_files:
    print(file)
```

If you want them sorted:

```python
txt_files = sorted(directory.glob("*.txt"))
```

`glob("*.txt")` looks only in that folder.  
`rglob("*.txt")` searches the folder and all subfolders.

If you want, I can also show how to filter out directories or return only filenames as strings.


## Summary

- Connected Context7 as an MCP server using `MCPServerStdio`.
- The agent automatically discovers MCP tools — no manual tool definitions needed.
- MCP servers and hand-written tools work together seamlessly.
- This pattern lets you extend your agent with any MCP-compatible server (databases, APIs, documentation, etc.).

## What You've Built

Across these 7 lessons, you built a complete coding assistant from scratch:

| Lesson | Tool | Capability |
|--------|------|------------|
| 1 | — | Basic chat |
| 2 | `read_file` | Read code files |
| 3 | `list_files` | Explore directories |
| 4 | `run_command` | Execute shell commands |
| 5 | `edit_file` | Create and modify files |
| 6 | `search_code` | Search codebase with patterns |
| 7 | Context7 MCP | Live documentation lookup |

> A coding agent is just **an LLM in a loop with tools**. The tools are simple Python functions. The magic is in the loop.